In [3]:
!pip install scipy

  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl (36.5 MB)


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

# --- YOUR LOCAL PATH ---
data_dir = "D:/wish-cycling/Data/archive (1)/dataset-resized"

img_size = (224, 224)
batch_size = 32

# 1. Training Generator (WITH Augmentation & 20% Split)
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2 # <--- This splits your main folder automatically
)

# 2. Validation Generator (NO Augmentation, just the 20% Split)
val_test_datagen = ImageDataGenerator(validation_split=0.2)

print("Loading Training Data...")
train_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training' # Grabs the 80% for training
)

print("\nLoading Validation Data...")
val_data = val_test_datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation' # Grabs the 20% for testing
)

# --- MODEL BUILDING ---
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation='softmax')(x) # 4 categories

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# --- CALLBACKS & SAVING ---
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

# Make sure the models folder exists
os.makedirs('models', exist_ok=True)

checkpoint = ModelCheckpoint(
    "models/waste_classifier.h5", # Saved exactly where your Streamlit app looks for it!
    monitor='val_accuracy',
    save_best_only=True
)

print("\nStarting Training...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

print("\n🎉 Training Complete! Best model saved to models/waste_classifier.h5")

Loading Training Data...
Found 1607 images belonging to 4 classes.

Loading Validation Data...
Found 400 images belonging to 4 classes.

Starting Training...
Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 778ms/step - accuracy: 0.6336 - loss: 0.8936

51/51 ━━━━━━━━━━━━━━━━━━━━ 59s 990ms/step - accuracy: 0.7598 - loss: 0.6276 - val_accuracy: 0.8350 - val_loss: 0.4155
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - accuracy: 0.8799 - loss: 0.3482

51/51 ━━━━━━━━━━━━━━━━━━━━ 48s 948ms/step - accuracy: 0.8793 - loss: 0.3310 - val_accuracy: 0.8575 - val_loss: 0.3783
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 787ms/step - accuracy: 0.9057 - loss: 0.2631

51/51 ━━━━━━━━━━━━━━━━━━━━ 48s 948ms/step - accuracy: 0.9098 - loss: 0.2497 - val_accuracy: 0.8775 - val_loss: 0.3544
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 47s 921ms/step - accuracy: 0.9235 - loss: 0.2142 - val_accuracy: 0.8750 - val_loss: 0.3551
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 770ms/step - accuracy: 0.9266 - loss: 0.1985

51/51 ━━━━━━━━━━━━━━━━━━━━ 48s 937ms/step - accuracy: 0.9340 - loss: 0.1868 - val_accuracy: 0.8800 - val_loss: 0.3495
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 794ms/step - accuracy: 0.9522 - loss: 0.1485

51/51 ━━━━━━━━━━━━━━━━━━━━ 49s 957ms/step - accuracy: 0.9452 - loss: 0.1555 - val_accuracy: 0.9025 - val_loss: 0.3502
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 48s 940ms/step - accuracy: 0.9583 - loss: 0.1317 - val_accuracy: 0.8775 - val_loss: 0.3170
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 49s 957ms/step - accuracy: 0.9589 - loss: 0.1165 - val_accuracy: 0.8900 - val_loss: 0.3213
Epoch 9/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 48s 938ms/step - accuracy: 0.9596 - loss: 0.1139 - val_accuracy: 0.8850 - val_loss: 0.3179

🎉 Training Complete! Best model saved to models/waste_classifier.h5


In [ ]:
!pip install --upgrade tensorflow

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached wrapt-2.1.2-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached grpcio-1.78.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached numpy-